In [ ]:
%env MUJOCO_GL=egl
import cv2
import torch
import numpy as np
from scipy.spatial.transform import Rotation
import mediapy

import mujoco
from gaussian_renderer import GSRendererMuJoCo

from gs_playground import ROOT_PATH

In [ ]:
# wxyz xyz
w2c = [0.3196, 0.6632, -0.609, 0.295, 0.0811, -0.1486, 1.2202]
# w2c = [0.0157989, 0.00645533, -0.00626437, 0.99983473, -0.0089715, -0.00186572, 0.516]

tmat = np.eye(4)
tmat[:3, :3] = Rotation.from_quat(w2c[:4], scalar_first=True).as_matrix()
tmat[:3, 3] = w2c[4:]

itmat = np.linalg.inv(tmat)
tvec = itmat[:3, 3]
cr = Rotation.from_matrix(itmat[:3, :3] @ Rotation.from_euler('x', 0, degrees=True).as_matrix())
quat_w = cr.as_quat(scalar_first=True) # wxyz
quat_x = cr.as_quat() # xyzw
print(f"Inverse Translation: {tvec}")
print(f"Inverse Quaternion (wxyz): {quat_w}")
print(f"Inverse Quaternion (xyzw): {quat_x}")

height = 480
fy = 603.62890625

fovy = 2 * np.arctan(height / (2 * fy))
fovy_deg = np.degrees(fovy)

print(f"FOVy (radians): {fovy}")
print(f"FOVy (degrees): {fovy_deg}")

# UR5e

In [ ]:
from gs_playground.src.manipulation.robots.universal_robots_ur5e_robotiq.ur5e_robotiq import UR5eRobotiq

_ASSETS_UR5E_DIR = ROOT_PATH / "models" / "robots" / "manipulation" / "universal_robots_ur5e_robotiq"
mjcf_path = _ASSETS_UR5E_DIR / "xmls/table30_02_stack_color_blocks.xml"

gaussians = UR5eRobotiq.robot_gaussians()
gaussians["background"] = UR5eRobotiq.robot_background_ply()


# Make model, data, and renderer
mj_model = mujoco.MjModel.from_xml_path(mjcf_path.as_posix())
mj_data = mujoco.MjData(mj_model)
mujoco.mj_resetDataKeyframe(mj_model, mj_data, 0)
mujoco.mj_forward(mj_model, mj_data)

H, W = 240, 320
renderer = mujoco.Renderer(mj_model, H, W)
renderer.update_scene(mj_data, 0)
img = renderer.render()

gsmj_renderer = GSRendererMuJoCo(gaussians, mj_model)
gsmj_renderer.update_gaussians(mj_data)
results = gsmj_renderer.render(mj_model, mj_data, list(range(mj_model.ncam)), W, H)
rgb = (255 * torch.clamp(results[0][0], 0., 1.)).to(torch.uint8).cpu().numpy()

mixed = cv2.addWeighted(img.astype(np.float32), 0.5, rgb.astype(np.float32), 0.5, 0).astype(np.uint8)

mediapy.show_image(np.hstack([img, mixed, rgb]))


# Franka

In [ ]:
from gs_playground.src.manipulation.robots.franka_emika_panda_robotiq.franka_robotiq import FrankaRobotiq

_ASSETS_FRANKA_DIR = ROOT_PATH / "models" / "robots" / "manipulation" / "franka_emika_panda_robotiq"
mjcf_path = _ASSETS_FRANKA_DIR / "xmls" / "table30_01_press_three_buttons.xml"

gaussians = FrankaRobotiq.robot_gaussians()
gaussians["background"] = FrankaRobotiq.robot_background_ply()

# Make model, data, and renderer
mj_model = mujoco.MjModel.from_xml_path(mjcf_path.as_posix())
mj_data = mujoco.MjData(mj_model)
mujoco.mj_resetDataKeyframe(mj_model, mj_data, 0)
mujoco.mj_forward(mj_model, mj_data)

H, W = 240, 320
renderer = mujoco.Renderer(mj_model, H, W)
renderer.update_scene(mj_data, 0)
img = renderer.render()

gsmj_renderer = GSRendererMuJoCo(gaussians, mj_model)
gsmj_renderer.update_gaussians(mj_data)
results = gsmj_renderer.render(mj_model, mj_data, list(range(mj_model.ncam)), W, H)
rgb = (255 * torch.clamp(results[0][0], 0., 1.)).to(torch.uint8).cpu().numpy()

mixed = cv2.addWeighted(img.astype(np.float32), 0.5, rgb.astype(np.float32), 0.5, 0).astype(np.uint8)

mediapy.show_image(np.hstack([img, mixed, rgb]), width=1080)
